# TASK 2.2: BENCHMARK BASELINE GỐC TRÊN TOÀN BỘ 18 TẬP CLEAN
### TÁI LẬP KẾT QUẢ CÔNG BỐ BÀI BÁO GỐC CVPR 2024 & THIẾT LẬP MỐC TRẦN HIỆU NĂNG

> **Người thực hiện**: Thành viên C *(Training & Evaluation Lead)*  
> **Môi trường**: Google Colab Pro GPU Tesla T4 / A100  
> **Hạ tầng lưu trữ**: Google Drive 5TB (`/content/drive/MyDrive/Fatformer`)  
> **Tệp kết quả đầu ra**: `/content/drive/MyDrive/Fatformer/log/baseline_clean_results.csv`  

---

### TIÊU CHUẨN NGHIỆM THU (DEFINITION OF DONE - DoD):
1. **Tái lập kết quả paper gốc CVPR 2024** với sai số không quá $\pm 0.5\%$:
   - Trung bình 8 kiến trúc GANs: $\approx \mathbf{98.4\%}$ ACC
   - Trung bình 10 kiến trúc Diffusion: $\approx \mathbf{95.0\%}$ ACC
   - Tập khó nhất (Guided Diffusion): $\approx \mathbf{76.1\%}$ ACC
2. **Tệp số liệu nghiệm thu**: Lưu an toàn trên Google Drive 5TB để bàn giao cho Thành viên B đưa vào Chương 1–2 của Báo cáo.

In [ ]:
# ================================================================================
# BƯỚC 1: KẾT NỐI GOOGLE DRIVE 5TB & ĐỒNG BỘ MÃ NGUỒN GIT TỪ NHÁNH HOANG
# ================================================================================
from google.colab import drive
import os, sys

print("[*] Đang mount Google Drive 5TB...")
drive.mount('/content/drive')
DRIVE_ROOT = "/content/drive/MyDrive/Fatformer"
assert os.path.exists(DRIVE_ROOT), f"LỖI: Không tìm thấy thư mục {DRIVE_ROOT}. Vui lòng kiểm tra lại cấu trúc Drive!"
print(f"[✓] Đã kết nối thành công kho lưu trữ: {DRIVE_ROOT}")

# Đồng bộ mã nguồn Git nhánh hoang
%cd /content
REPO_DIR = "/content/fatformer-xla"
if not os.path.exists(REPO_DIR):
    print("[*] Đang clone repository từ branch hoang...")
    !git clone -b hoang https://github.com/LeeVHoangtk3/fatformer-xla.git
else:
    print("[*] Repository đã tồn tại, đang fetch và pull nhánh hoang mới nhất...")
    %cd {REPO_DIR}
    !git fetch origin hoang
    !git checkout hoang
    !git pull origin hoang

%cd {REPO_DIR}
print(f"[✓] Thư mục làm việc hiện tại: {os.getcwd()}")

In [ ]:
# ================================================================================
# BƯỚC 2: CÀI ĐẶT CÁC THƯ VIỆN BỔ TRỢ CẦN THIẾT
# ================================================================================
!pip install -q timm ftfy regex scikit-learn pandas tqdm

In [ ]:
# ================================================================================
# BƯỚC 3: KIỂM TRA PHẦN CỨNG & TÀI NGUYÊN GPU TRÊN COLAB
# ================================================================================
import torch

print("=" * 80)
print("THÔNG SỐ PHẦN CỨNG COLAB PRO")
print("=" * 80)
print(f"• Phiên bản PyTorch: {torch.__version__}")
print(f"• CUDA Sẵn sàng:    {torch.cuda.is_available()}")
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"• Thiết bị GPU:     {device_name} ({vram_gb:.2f} GB VRAM)")
else:
    print("[CẢNH BÁO] Không tìm thấy GPU! Vui lòng vào Runtime > Change runtime type > Chọn GPU (T4 hoặc A100).")
print("=" * 80)

In [ ]:
# ================================================================================
# BƯỚC 4: GIẢI NÉN CHỐNG NGHẼN I/O 18 TẬP TEST CLEAN SANG Ổ SSD NVME CỦA COLAB
# ================================================================================
import os, tarfile, time

LOCAL_TEST_ROOT = "/content/dataset_local/test_clean"
os.makedirs(LOCAL_TEST_ROOT, exist_ok=True)

# 1. Giải nén 10 tập Diffusion (1.22 GB - Tốc độ đọc GCP ~10-15 giây)
diff_tar = f"{DRIVE_ROOT}/datasets/test_benchmark_diffusion.tar"
if os.path.exists(diff_tar):
    print(f"[*] Đang giải nén 10 tập Diffusion từ {diff_tar}...")
    t0 = time.time()
    with tarfile.open(diff_tar, "r") as tar:
        tar.extractall(LOCAL_TEST_ROOT)
    print(f"  -> [✓] Hoàn tất 10 tập Diffusion trong {time.time() - t0:.1f}s!")
else:
    print(f"[!] CẢNH BÁO: Không tìm thấy file {diff_tar}")

# 2. Giải nén 8 tập GANs (18.74 GB - Tốc độ đọc GCP ~2-3 phút)
gans_tar = f"{DRIVE_ROOT}/datasets/test_benchmark_gans.tar"
if os.path.exists(gans_tar):
    print(f"[*] Đang giải nén 8 tập GANs từ {gans_tar}...")
    t0 = time.time()
    with tarfile.open(gans_tar, "r") as tar:
        tar.extractall(LOCAL_TEST_ROOT)
    print(f"  -> [✓] Hoàn tất 8 tập GANs trong {time.time() - t0:.1f}s!")
else:
    print(f"[!] CẢNH BÁO: Không tìm thấy file {gans_tar}")

unpacked_subsets = sorted(os.listdir(LOCAL_TEST_ROOT))
print(f"\n[✓] Tổng cộng đã sẵn sàng {len(unpacked_subsets)} tập test trên SSD NVMe local:")
for s in unpacked_subsets:
    print(f"  - {s}")

### BƯỚC 5: [TÙY CHỌN] KIỂM THỬ NHANH VỚI FAST-EVAL (< 8 PHÚT)
Nếu muốn rà soát nhanh toàn bộ pipeline trước khi chạy đánh giá 100% mẫu, bạn có thể chạy cell này.
Mỗi tập test sẽ lấy mẫu cân bằng **500 ảnh (250 real + 250 fake, seed=42)**.

In [ ]:
# [TÙY CHỌN] Chạy Fast-Eval 500 ảnh/tập test (< 8 phút)
!python tools/fast_eval.py \
    --checkpoint /content/drive/MyDrive/Fatformer/pretrained/fatformer_4class_ckpt.pth \
    --clip_path /content/drive/MyDrive/Fatformer/pretrained/ViT-L-14.pt \
    --dataset_root /content/dataset_local/test_clean \
    --subsets all \
    --samples_per_subset 500 \
    --batch_size 32 \
    --device cuda \
    --output_csv /content/drive/MyDrive/Fatformer/log/fast_eval_clean_baseline.csv

### BƯỚC 6: [CHÍNH THỨC - DoD TASK 2.2] FULL BENCHMARK 100% SỐ ẢNH TRÊN 18 TẬP TEST
Cell này thực thi script [`tools/full_eval.py`](file:///d:/GIT%20REPO/.nam4/fatformer-xla/tools/full_eval.py):
- Đánh giá toàn bộ 100% số ảnh của 18 tập test.
- Tự động phân nhóm GANs (8 tập) và Diffusion (10 tập), tính riêng `MEAN_GANS` và `MEAN_DIFFUSION`.
- Tự động thẩm định sai số đối chiếu với bài báo CVPR 2024 (DoD: $\le \pm 0.5\%$).
- Tự động xuất kết quả sang Google Drive 5TB: `/content/drive/MyDrive/Fatformer/log/baseline_clean_results.csv`.

In [ ]:
# [CHÍNH THỨC] FULL BENCHMARK BASELINE CLEAN TOÀN BỘ 18 TẬP TEST
!python tools/full_eval.py \
    --checkpoint /content/drive/MyDrive/Fatformer/pretrained/fatformer_4class_ckpt.pth \
    --clip_path /content/drive/MyDrive/Fatformer/pretrained/ViT-L-14.pt \
    --test_root /content/dataset_local/test_clean \
    --subsets all \
    --batch_size 32 \
    --num_workers 4 \
    --device cuda \
    --output_csv /content/drive/MyDrive/Fatformer/log/baseline_clean_results.csv \
    --output_markdown /content/drive/MyDrive/Fatformer/log/baseline_clean_results.md

### BƯỚC 7: HIỂN THỊ BẢNG KẾT QUẢ ĐÃ LƯU TRÊN GOOGLE DRIVE 5TB

In [ ]:
import os
import pandas as pd
from IPython.display import display

csv_path = "/content/drive/MyDrive/Fatformer/log/baseline_clean_results.csv"
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print("=" * 86)
    print("[✓] TỆP KẾT QUẢ ĐÃ LƯU TRÊN GOOGLE DRIVE 5TB:")
    print(csv_path)
    print("=" * 86)
    display(df)
else:
    print(f"[!] Chưa tìm thấy file CSV tại: {csv_path}. Vui lòng chạy Bước 6 trước!")